<p align="right"><a href="./Building_your_Deep_Neural_Network_Step_by_Step.ipynb">English</a> | <strong>中文</strong></p>

# 一步步构建你的深度神经网络（Building your Deep Neural Network: Step by Step）

欢迎来到你第 4 周的作业（共两部分中的第 1 部分）！之前你训练了一个带单个隐藏层的 2 层神经网络。这一周，你将构建一个想要多少层就有多少层的深度神经网络！

- 在这个 notebook 里，你会实现构建一个深度神经网络所需的所有函数。
- 在下一个作业里，你会用这些函数来构建一个用于图像分类的深度神经网络。

**到本作业结束时，你将能够：**

- 使用像 ReLU 这样的非线性单元来改进你的模型
- 构建一个更深的神经网络（多于 1 个隐藏层）
- 实现一个易于使用的神经网络类

**记号**：
- 上标 $[l]$ 表示与第 $l$ 层相关的量。
    - 例子：$a^{[L]}$ 是第 $L$ 层的激活。$W^{[L]}$ 和 $b^{[L]}$ 是第 $L$ 层的参数。
- 上标 $(i)$ 表示与第 $i$ 个样本相关的量。
    - 例子：$x^{(i)}$ 是第 $i$ 个训练样本。
- 下标 $i$ 表示一个向量的第 $i$ 个元素。
    - 例子：$a^{[l]}_i$ 表示第 $l$ 层激活的第 $i$ 个元素。

我们开始吧！

## 关于提交给 AutoGrader 的重要提示

在把你的作业提交给 AutoGrader 之前，请确认你没有做以下这些事：

1. 你没有在作业里加任何 _额外的_ `print` 语句。
2. 你没有在作业里加任何 _额外的_ code cell。
3. 你没有更改任何函数参数。
4. 你没有在评分练习内部使用任何全局变量。除非题目特别要求，否则请避免这样做，改用局部变量。
5. 你没有在不需要改的地方更改作业代码，比如创建 _额外的_ 变量。

如果你做了上述任何一件事，提交作业时你会得到类似 `Grader Error: Grader feedback not found`（或其他类似意外）的错误。在寻求帮助/调试作业里的错误之前，请先检查这些。如果确实如此、而你又不记得自己改了什么，可以按这些[说明](https://www.coursera.org/learn/neural-networks-deep-learning/supplement/iLwon/h-ow-to-refresh-your-workspace)获取一份全新的作业副本。

## 目录
- [1 - 库 Packages](#1)
- [2 - 概览 Outline](#2)
- [3 - 初始化 Initialization](#3)
    - [3.1 - 2 层神经网络](#3-1)
        - [练习 Exercise 1 - initialize_parameters](#ex-1)
    - [3.2 - L 层神经网络](#3-2)
        - [练习 Exercise 2 - initialize_parameters_deep](#ex-2)
- [4 - 前向传播模块](#4)
    - [4.1 - Linear Forward](#4-1)
        - [练习 Exercise 3 - linear_forward](#ex-3)
    - [4.2 - Linear-Activation Forward](#4-2)
        - [练习 Exercise 4 - linear_activation_forward](#ex-4)
    - [4.3 - L 层模型](#4-3)
        - [练习 Exercise 5 - L_model_forward](#ex-5)
- [5 - 代价函数](#5)
    - [练习 Exercise 6 - compute_cost](#ex-6)
- [6 - 反向传播模块](#6)
    - [6.1 - Linear Backward](#6-1)
        - [练习 Exercise 7 - linear_backward](#ex-7)
    - [6.2 - Linear-Activation Backward](#6-2)
        - [练习 Exercise 8 - linear_activation_backward](#ex-8)
    - [6.3 - L-Model Backward](#6-3)
        - [练习 Exercise 9 - L_model_backward](#ex-9)
    - [6.4 - 更新参数](#6-4)
        - [练习 Exercise 10 - update_parameters](#ex-10)

<a name='1'></a>
## 1 - 库 Packages

首先，导入本作业需要的所有库。

- [numpy](www.numpy.org) 是 Python 中科学计算的主要库。
- [matplotlib](http://matplotlib.org) 是 Python 中绘图的库。
- dnn_utils 为本 notebook 提供了一些必要的函数。
- testCases 提供了一些测试用例，用来评估你函数的正确性
- np.random.seed(1) 用来让所有随机函数调用保持一致。它有助于给你的作业评分。请不要更改这个 seed！

In [1]:
### v1.2

In [2]:
import numpy as np
import h5py
import matplotlib.pyplot as plt
from testCases import *
from dnn_utils import sigmoid, sigmoid_backward, relu, relu_backward
from public_tests import *

import copy
%matplotlib inline
plt.rcParams['figure.figsize'] = (5.0, 4.0) # set default size of plots
plt.rcParams['image.interpolation'] = 'nearest'
plt.rcParams['image.cmap'] = 'gray'

%load_ext autoreload
%autoreload 2

np.random.seed(1)

<a name='2'></a>
## 2 - 概览 Outline

要构建你的神经网络，你会实现若干 "辅助函数（helper functions）"。这些辅助函数会在下一个作业里用来构建一个两层神经网络和一个 L 层神经网络。

每个小的辅助函数都会有详细说明，一步步带你完成必要的步骤。这是本作业各步骤的概览：

- 为一个两层网络和一个 $L$ 层神经网络初始化参数
- 实现前向传播模块（下图中紫色部分所示）
     - 完成某一层前向传播步骤的 LINEAR 部分（得到 $Z^{[l]}$）。
     - ACTIVATION 函数已为你提供（relu/sigmoid）
     - 把前两步合并进一个新的 [LINEAR->ACTIVATION] 前向函数。
     - 把 [LINEAR->RELU] 前向函数堆叠 L-1 次（对第 1 到 L-1 层），并在最后加一个 [LINEAR->SIGMOID]（对最后一层 $L$）。这就给了你一个新的 L_model_forward 函数。
- 计算损失
- 实现反向传播模块（下图中红色部分所示）
    - 完成某一层反向传播步骤的 LINEAR 部分
    - ACTIVATION 函数的梯度已为你提供（relu_backward/sigmoid_backward）
    - 把前两步合并进一个新的 [LINEAR->ACTIVATION] 反向函数
    - 在一个新的 L_model_backward 函数里，把 [LINEAR->RELU] 反向堆叠 L-1 次，并加上 [LINEAR->SIGMOID] 反向
- 最后，更新参数

<img src="images/final outline.png" style="width:800px;height:500px;">
<caption><center><b>图 1</b></center></caption><br>


**注意**：

对每个前向函数，都有一个对应的反向函数。这就是为什么在你前向模块的每一步，你都会把一些值存进一个 cache。这些缓存的值对计算梯度很有用。

在反向传播模块里，你随后就能用这个 cache 来计算梯度。别担心，本作业会准确地告诉你如何执行这每一步！

<a name='3'></a>
## 3 - 初始化 Initialization

你将写两个辅助函数来为你的模型初始化参数。第一个函数用来为一个两层模型初始化参数。第二个把这个初始化过程推广到 $L$ 层。

<a name='3-1'></a>
### 3.1 - 2 层神经网络

<a name='ex-1'></a>
### 练习 Exercise 1 - initialize_parameters

创建并初始化这个 2 层神经网络的参数。

**说明**：

- 模型结构是：*LINEAR -> RELU -> LINEAR -> SIGMOID*。
- 对权重矩阵用这个随机初始化：`np.random.randn(d0, d1, ..., dn) * 0.01`，形状要正确。[np.random.randn](https://numpy.org/doc/stable/reference/random/generated/numpy.random.randn.html) 的文档
- 对偏置用零初始化：`np.zeros(shape)`。[np.zeros](https://numpy.org/doc/stable/reference/generated/numpy.zeros.html) 的文档

In [7]:
# GRADED FUNCTION: initialize_parameters

def initialize_parameters(n_x, n_h, n_y):
    """
    Argument:
    n_x -- size of the input layer
    n_h -- size of the hidden layer
    n_y -- size of the output layer
    
    Returns:
    parameters -- python dictionary containing your parameters:
                    W1 -- weight matrix of shape (n_h, n_x)
                    b1 -- bias vector of shape (n_h, 1)
                    W2 -- weight matrix of shape (n_y, n_h)
                    b2 -- bias vector of shape (n_y, 1)
    """
    
    np.random.seed(1)
    
    #(≈ 4 lines of code)
    # W1 = ...
    # b1 = ...
    # W2 = ...
    # b2 = ...
    # YOUR CODE STARTS HERE
    W1 = np.random.randn(n_h,n_x) * 0.01
    W2 = np.random.randn(n_y,n_h) * 0.01
    b1 = np.zeros((n_h,1))
    b2 = np.zeros((n_y,1))
    # YOUR CODE ENDS HERE
    
    parameters = {"W1": W1,
                  "b1": b1,
                  "W2": W2,
                  "b2": b2}
    
    return parameters    

In [8]:
print("Test Case 1:\n")
parameters = initialize_parameters(3,2,1)

print("W1 = " + str(parameters["W1"]))
print("b1 = " + str(parameters["b1"]))
print("W2 = " + str(parameters["W2"]))
print("b2 = " + str(parameters["b2"]))

initialize_parameters_test_1(initialize_parameters)

print("\033[90m\nTest Case 2:\n")
parameters = initialize_parameters(4,3,2)

print("W1 = " + str(parameters["W1"]))
print("b1 = " + str(parameters["b1"]))
print("W2 = " + str(parameters["W2"]))
print("b2 = " + str(parameters["b2"]))

initialize_parameters_test_2(initialize_parameters)

Test Case 1:

W1 = [[ 0.01624345 -0.00611756 -0.00528172]
 [-0.01072969  0.00865408 -0.02301539]]
b1 = [[0.]
 [0.]]
W2 = [[ 0.01744812 -0.00761207]]
b2 = [[0.]]
 All tests passed.

Test Case 2:

W1 = [[ 0.01624345 -0.00611756 -0.00528172 -0.01072969]
 [ 0.00865408 -0.02301539  0.01744812 -0.00761207]
 [ 0.00319039 -0.0024937   0.01462108 -0.02060141]]
b1 = [[0.]
 [0.]
 [0.]]
W2 = [[-0.00322417 -0.00384054  0.01133769]
 [-0.01099891 -0.00172428 -0.00877858]]
b2 = [[0.]
 [0.]]
 All tests passed.


***期望输出***
```
Test Case 1:

W1 = [[ 0.01624345 -0.00611756 -0.00528172]
 [-0.01072969  0.00865408 -0.02301539]]
b1 = [[0.]
 [0.]]
W2 = [[ 0.01744812 -0.00761207]]
b2 = [[0.]]
 All tests passed.

Test Case 2:

W1 = [[ 0.01624345 -0.00611756 -0.00528172 -0.01072969]
 [ 0.00865408 -0.02301539  0.01744812 -0.00761207]
 [ 0.00319039 -0.0024937   0.01462108 -0.02060141]]
b1 = [[0.]
 [0.]
 [0.]]
W2 = [[-0.00322417 -0.00384054  0.01133769]
 [-0.01099891 -0.00172428 -0.00877858]]
b2 = [[0.]
 [0.]]
 All tests passed.
```

<a name='3-2'></a>
### 3.2 - L 层神经网络

更深的 L 层神经网络的初始化更复杂，因为有多得多的权重矩阵和偏置向量。在完成 `initialize_parameters_deep` 函数时，你应确保各层之间的维度匹配。回忆一下 $n^{[l]}$ 是第 $l$ 层的单元数。例如，若你输入 $X$ 的大小是 $(12288, 209)$（有 $m=209$ 个样本），那么：

<table style="width:100%">
    <tr>
        <td>  </td>
        <td> <b>Shape of W</b> </td>
        <td> <b>Shape of b</b>  </td>
        <td> <b>Activation</b> </td>
        <td> <b>Shape of Activation</b> </td>
    <tr>
    <tr>
        <td> <b>Layer 1</b> </td>
        <td> $(n^{[1]},12288)$ </td>
        <td> $(n^{[1]},1)$ </td>
        <td> $Z^{[1]} = W^{[1]}  X + b^{[1]} $ </td>
        <td> $(n^{[1]},209)$ </td>
    <tr>
    <tr>
        <td> <b>Layer 2</b> </td>
        <td> $(n^{[2]}, n^{[1]})$  </td>
        <td> $(n^{[2]},1)$ </td>
        <td>$Z^{[2]} = W^{[2]} A^{[1]} + b^{[2]}$ </td>
        <td> $(n^{[2]}, 209)$ </td>
    <tr>
       <tr>
        <td> $\vdots$ </td>
        <td> $\vdots$  </td>
        <td> $\vdots$  </td>
        <td> $\vdots$</td>
        <td> $\vdots$  </td>
    <tr>
   <tr>
       <td> <b>Layer L-1</b> </td>
        <td> $(n^{[L-1]}, n^{[L-2]})$ </td>
        <td> $(n^{[L-1]}, 1)$  </td>
        <td>$Z^{[L-1]} =  W^{[L-1]} A^{[L-2]} + b^{[L-1]}$ </td>
        <td> $(n^{[L-1]}, 209)$ </td>
   <tr>
   <tr>
       <td> <b>Layer L</b> </td>
        <td> $(n^{[L]}, n^{[L-1]})$ </td>
        <td> $(n^{[L]}, 1)$ </td>
        <td> $Z^{[L]} =  W^{[L]} A^{[L-1]} + b^{[L]}$</td>
        <td> $(n^{[L]}, 209)$  </td>
    <tr>
</table>

记住，当你在 python 里计算 $W X + b$ 时，它会执行广播。例如，若：

$$ W = \begin{bmatrix}
    w_{00}  & w_{01} & w_{02} \\
    w_{10}  & w_{11} & w_{12} \\
    w_{20}  & w_{21} & w_{22}
\end{bmatrix}\;\;\; X = \begin{bmatrix}
    x_{00}  & x_{01} & x_{02} \\
    x_{10}  & x_{11} & x_{12} \\
    x_{20}  & x_{21} & x_{22}
\end{bmatrix} \;\;\; b =\begin{bmatrix}
    b_0  \\
    b_1  \\
    b_2
\end{bmatrix}\tag{2}$$

那么 $WX + b$ 会是：

$$ WX + b = \begin{bmatrix}
    (w_{00}x_{00} + w_{01}x_{10} + w_{02}x_{20}) + b_0 & (w_{00}x_{01} + w_{01}x_{11} + w_{02}x_{21}) + b_0 & \cdots \\
    (w_{10}x_{00} + w_{11}x_{10} + w_{12}x_{20}) + b_1 & (w_{10}x_{01} + w_{11}x_{11} + w_{12}x_{21}) + b_1 & \cdots \\
    (w_{20}x_{00} + w_{21}x_{10} + w_{22}x_{20}) + b_2 &  (w_{20}x_{01} + w_{21}x_{11} + w_{22}x_{21}) + b_2 & \cdots
\end{bmatrix}\tag{3}  $$

<a name='ex-2'></a>
### 练习 Exercise 2 - initialize_parameters_deep

为一个 L 层神经网络实现初始化。

**说明**：
- 模型结构是 *[LINEAR -> RELU] $ \times$ (L-1) -> LINEAR -> SIGMOID*。即，它有 $L-1$ 层用 ReLU 激活函数，后接一个用 sigmoid 激活函数的输出层。
- 对权重矩阵用随机初始化。用 `np.random.randn(d0, d1, ..., dn) * 0.01`。
- 对偏置用零初始化。用 `np.zeros(shape)`。
- 你会把 $n^{[l]}$（不同层的单元数）存进一个变量 `layer_dims`。例如，上周平面数据分类模型的 `layer_dims` 会是 [2,4,1]：有两个输入、一个含 4 个隐藏单元的隐藏层、一个含 1 个输出单元的输出层。这意味着 `W1` 的形状是 (4,2)、`b1` 是 (4,1)、`W2` 是 (1,4)、`b2` 是 (1,1)。现在你要把它推广到 $L$ 层！
- 这里是 $L=1$（单层神经网络）的实现。它应该能启发你实现一般情形（L 层神经网络）。
```python
    if L == 1:
        parameters["W" + str(L)] = np.random.randn(layer_dims[1], layer_dims[0]) * 0.01
        parameters["b" + str(L)] = np.zeros((layer_dims[1], 1))
```

In [11]:
# GRADED FUNCTION: initialize_parameters_deep

def initialize_parameters_deep(layer_dims):
    """
    Arguments:
    layer_dims -- python array (list) containing the dimensions of each layer in our network
    
    Returns:
    parameters -- python dictionary containing your parameters "W1", "b1", ..., "WL", "bL":
                    Wl -- weight matrix of shape (layer_dims[l], layer_dims[l-1])
                    bl -- bias vector of shape (layer_dims[l], 1)
    """
    
    np.random.seed(3)
    parameters = {}
    L = len(layer_dims) # number of layers in the network

    for l in range(1, L):
        #(≈ 2 lines of code)
        # parameters['W' + str(l)] = ...
        # parameters['b' + str(l)] = ...
        # YOUR CODE STARTS HERE
        parameters["W" + str(l)] = np.random.randn(layer_dims[l],layer_dims[l-1]) * 0.01
        parameters["b" + str(l)] = np.zeros((layer_dims[l],1))
        # YOUR CODE ENDS HERE
        
        assert(parameters['W' + str(l)].shape == (layer_dims[l], layer_dims[l - 1]))
        assert(parameters['b' + str(l)].shape == (layer_dims[l], 1))

        
    return parameters

In [12]:
print("Test Case 1:\n")
parameters = initialize_parameters_deep([5,4,3])

print("W1 = " + str(parameters["W1"]))
print("b1 = " + str(parameters["b1"]))
print("W2 = " + str(parameters["W2"]))
print("b2 = " + str(parameters["b2"]))

initialize_parameters_deep_test_1(initialize_parameters_deep)

print("\033[90m\nTest Case 2:\n")
parameters = initialize_parameters_deep([4,3,2])

print("W1 = " + str(parameters["W1"]))
print("b1 = " + str(parameters["b1"]))
print("W2 = " + str(parameters["W2"]))
print("b2 = " + str(parameters["b2"]))
initialize_parameters_deep_test_2(initialize_parameters_deep)

Test Case 1:

W1 = [[ 0.01788628  0.0043651   0.00096497 -0.01863493 -0.00277388]
 [-0.00354759 -0.00082741 -0.00627001 -0.00043818 -0.00477218]
 [-0.01313865  0.00884622  0.00881318  0.01709573  0.00050034]
 [-0.00404677 -0.0054536  -0.01546477  0.00982367 -0.01101068]]
b1 = [[0.]
 [0.]
 [0.]
 [0.]]
W2 = [[-0.01185047 -0.0020565   0.01486148  0.00236716]
 [-0.01023785 -0.00712993  0.00625245 -0.00160513]
 [-0.00768836 -0.00230031  0.00745056  0.01976111]]
b2 = [[0.]
 [0.]
 [0.]]
 All tests passed.

Test Case 2:

W1 = [[ 0.01788628  0.0043651   0.00096497 -0.01863493]
 [-0.00277388 -0.00354759 -0.00082741 -0.00627001]
 [-0.00043818 -0.00477218 -0.01313865  0.00884622]]
b1 = [[0.]
 [0.]
 [0.]]
W2 = [[ 0.00881318  0.01709573  0.00050034]
 [-0.00404677 -0.0054536  -0.01546477]]
b2 = [[0.]
 [0.]]
 All tests passed.


***期望输出***
```
Test Case 1:

W1 = [[ 0.01788628  0.0043651   0.00096497 -0.01863493 -0.00277388]
 [-0.00354759 -0.00082741 -0.00627001 -0.00043818 -0.00477218]
 [-0.01313865  0.00884622  0.00881318  0.01709573  0.00050034]
 [-0.00404677 -0.0054536  -0.01546477  0.00982367 -0.01101068]]
b1 = [[0.]
 [0.]
 [0.]
 [0.]]
W2 = [[-0.01185047 -0.0020565   0.01486148  0.00236716]
 [-0.01023785 -0.00712993  0.00625245 -0.00160513]
 [-0.00768836 -0.00230031  0.00745056  0.01976111]]
b2 = [[0.]
 [0.]
 [0.]]
 All tests passed.

Test Case 2:

W1 = [[ 0.01788628  0.0043651   0.00096497 -0.01863493]
 [-0.00277388 -0.00354759 -0.00082741 -0.00627001]
 [-0.00043818 -0.00477218 -0.01313865  0.00884622]]
b1 = [[0.]
 [0.]
 [0.]]
W2 = [[ 0.00881318  0.01709573  0.00050034]
 [-0.00404677 -0.0054536  -0.01546477]]
b2 = [[0.]
 [0.]]
 All tests passed.
```

<a name='4'></a>
## 4 - 前向传播模块

<a name='4-1'></a>
### 4.1 - Linear Forward

现在你已经初始化了参数，可以来做前向传播模块了。先实现一些基础函数，稍后实现模型时可以再次用到。现在，你会按以下顺序完成三个函数：

- LINEAR
- LINEAR -> ACTIVATION，其中 ACTIVATION 会是 ReLU 或 Sigmoid。
- [LINEAR -> RELU] $\times$ (L-1) -> LINEAR -> SIGMOID（整个模型）

线性前向模块（对所有样本向量化）计算以下方程：

$$Z^{[l]} = W^{[l]}A^{[l-1]} +b^{[l]}\tag{4}$$

其中 $A^{[0]} = X$。

<a name='ex-3'></a>
### 练习 Exercise 3 - linear_forward

构建前向传播的线性部分。

**提醒**：
这个单元的数学表示是 $Z^{[l]} = W^{[l]}A^{[l-1]} +b^{[l]}$。你可能也会发现 `np.dot()` 有用。如果你的维度不匹配，打印 `W.shape` 可能有帮助。

In [15]:
# GRADED FUNCTION: linear_forward

def linear_forward(A, W, b):
    """
    Implement the linear part of a layer's forward propagation.

    Arguments:
    A -- activations from previous layer (or input data): (size of previous layer, number of examples)
    W -- weights matrix: numpy array of shape (size of current layer, size of previous layer)
    b -- bias vector, numpy array of shape (size of the current layer, 1)

    Returns:
    Z -- the input of the activation function, also called pre-activation parameter 
    cache -- a python tuple containing "A", "W" and "b" ; stored for computing the backward pass efficiently
    """
    
    #(≈ 1 line of code)
    # Z = ...
    # YOUR CODE STARTS HERE
    Z = np.dot(W,A) + b

    # YOUR CODE ENDS HERE
    cache = (A, W, b)
    
    return Z, cache

In [16]:
t_A, t_W, t_b = linear_forward_test_case()
t_Z, t_linear_cache = linear_forward(t_A, t_W, t_b)
print("Z = " + str(t_Z))

linear_forward_test(linear_forward)

Z = [[ 3.26295337 -1.23429987]]
 All tests passed.


***期望输出***
```
Z = [[ 3.26295337 -1.23429987]]
```

<a name='4-2'></a>
### 4.2 - Linear-Activation Forward

在这个 notebook 里，你会用两个激活函数：

- **Sigmoid**：$\sigma(Z) = \sigma(W A + b) = \frac{1}{ 1 + e^{-(W A + b)}}$。已为你提供 `sigmoid` 函数，它返回**两**样东西：激活值 "`a`" 和一个包含 "`Z`" 的 "`cache`"（它就是我们将喂给对应反向函数的东西）。要用它，你只需调用：
``` python
A, activation_cache = sigmoid(Z)
```

- **ReLU**：ReLu 的数学公式是 $A = RELU(Z) = max(0, Z)$。已为你提供 `relu` 函数。这个函数返回**两**样东西：激活值 "`A`" 和一个包含 "`Z`" 的 "`cache`"（它就是你将喂给对应反向函数的东西）。要用它，你只需调用：
``` python
A, activation_cache = relu(Z)
```

为进一步方便，你要把两个函数（Linear 和 Activation）组合成一个函数（LINEAR->ACTIVATION）。因此，你会实现一个函数：先做 LINEAR 前向步骤，再接一个 ACTIVATION 前向步骤。

<a name='ex-4'></a>
### 练习 Exercise 4 - linear_activation_forward

实现 *LINEAR->ACTIVATION* 层的前向传播。数学关系是：$A^{[l]} = g(Z^{[l]}) = g(W^{[l]}A^{[l-1]} +b^{[l]})$，其中激活 "g" 可以是 sigmoid() 或 relu()。使用 `linear_forward()` 和正确的激活函数。

In [25]:
# GRADED FUNCTION: linear_activation_forward

def linear_activation_forward(A_prev, W, b, activation):
    """
    Implement the forward propagation for the LINEAR->ACTIVATION layer

    Arguments:
    A_prev -- activations from previous layer (or input data): (size of previous layer, number of examples)
    W -- weights matrix: numpy array of shape (size of current layer, size of previous layer)
    b -- bias vector, numpy array of shape (size of the current layer, 1)
    activation -- the activation to be used in this layer, stored as a text string: "sigmoid" or "relu"

    Returns:
    A -- the output of the activation function, also called the post-activation value 
    cache -- a python tuple containing "linear_cache" and "activation_cache";
             stored for computing the backward pass efficiently
    """
    
    if activation == "sigmoid":
        #(≈ 2 lines of code)
        # Z, linear_cache = ...
        # A, activation_cache = ...
        # YOUR CODE STARTS HERE
        Z, linear_cache = linear_forward(A_prev,W,b)
        A, activation_cache = sigmoid(Z)
        # YOUR CODE ENDS HERE
    
    elif activation == "relu":
        #(≈ 2 lines of code)
        # Z, linear_cache = ...
        # A, activation_cache = ...
        # YOUR CODE STARTS HERE
        Z, linear_cache = linear_forward(A_prev,W,b)
        A, activation_cache = relu(Z)
        # YOUR CODE ENDS HERE
    cache = (linear_cache, activation_cache)

    return A, cache

In [26]:
t_A_prev, t_W, t_b = linear_activation_forward_test_case()

t_A, t_linear_activation_cache = linear_activation_forward(t_A_prev, t_W, t_b, activation = "sigmoid")
print("With sigmoid: A = " + str(t_A))

t_A, t_linear_activation_cache = linear_activation_forward(t_A_prev, t_W, t_b, activation = "relu")
print("With ReLU: A = " + str(t_A))

linear_activation_forward_test(linear_activation_forward)

With sigmoid: A = [[0.96890023 0.11013289]]
With ReLU: A = [[3.43896131 0.        ]]
 All tests passed.


***期望输出***
```
With sigmoid: A = [[0.96890023 0.11013289]]
With ReLU: A = [[3.43896131 0.        ]]
```

**注意**：在深度学习里，"[LINEAR->ACTIVATION]" 的计算被算作神经网络里的单独一层，而不是两层。

<a name='4-3'></a>
### 4.3 - L 层模型

为了在实现 $L$ 层神经网络时*更加*方便，你会需要一个函数，把前一个函数（带 RELU 的 `linear_activation_forward`）复制 $L-1$ 次，然后再接一个带 SIGMOID 的 `linear_activation_forward`。

<img src="images/model_architecture_kiank.png" style="width:600px;height:300px;">
<caption><center> <b>图 2</b>：*[LINEAR -> RELU] $\times$ (L-1) -> LINEAR -> SIGMOID* 模型</center></caption><br>

<a name='ex-5'></a>
### 练习 Exercise 5 - L_model_forward

实现上述模型的前向传播。

**说明**：在下面的代码里，变量 `AL` 将表示 $A^{[L]} = \sigma(Z^{[L]}) = \sigma(W^{[L]} A^{[L-1]} + b^{[L]})$。（这有时也叫 `Yhat`，即这就是 $\hat{Y}$。）

**提示**：
- 使用你之前写的那些函数
- 用一个 for 循环把 [LINEAR->RELU] 复制 (L-1) 次
- 别忘了把 caches 记录在 "caches" 列表里。要往一个 `list` 里加一个新值 `c`，你可以用 `list.append(c)`。

In [27]:
# GRADED FUNCTION: L_model_forward

def L_model_forward(X, parameters):
    """
    Implement forward propagation for the [LINEAR->RELU]*(L-1)->LINEAR->SIGMOID computation
    
    Arguments:
    X -- data, numpy array of shape (input size, number of examples)
    parameters -- output of initialize_parameters_deep()
    
    Returns:
    AL -- activation value from the output (last) layer
    caches -- list of caches containing:
                every cache of linear_activation_forward() (there are L of them, indexed from 0 to L-1)
    """

    caches = []
    A = X
    L = len(parameters) // 2                  # number of layers in the neural network
    
    # Implement [LINEAR -> RELU]*(L-1). Add "cache" to the "caches" list.
    # The for loop starts at 1 because layer 0 is the input
    for l in range(1, L):
        A_prev = A 
        #(≈ 2 lines of code)
        # A, cache = ...
        # caches ...
        # YOUR CODE STARTS HERE
        A, cache = linear_activation_forward(A_prev,parameters["W"+str(l)],parameters["b"+str(l)],activation="relu")
        caches.append(cache)
        # YOUR CODE ENDS HERE
    
    # Implement LINEAR -> SIGMOID. Add "cache" to the "caches" list.
    #(≈ 2 lines of code)
    # AL, cache = ...
    # caches ...
    # YOUR CODE STARTS HERE
    AL,cache = linear_activation_forward(A,parameters["W"+str(L)],parameters["b"+str(L)],activation="sigmoid")
    caches.append(cache)
    # YOUR CODE ENDS HERE
          
    return AL, caches

In [28]:
t_X, t_parameters = L_model_forward_test_case_2hidden()
t_AL, t_caches = L_model_forward(t_X, t_parameters)

print("AL = " + str(t_AL))

L_model_forward_test(L_model_forward)

AL = [[0.03921668 0.70498921 0.19734387 0.04728177]]
 All tests passed.


***期望输出***
```
AL = [[0.03921668 0.70498921 0.19734387 0.04728177]]
```

**太棒了！** 你实现了一个完整的前向传播：它接收输入 X，输出一个包含你预测的行向量 $A^{[L]}$。它还把所有中间值记录在 "caches" 里。用 $A^{[L]}$，你就能计算你预测的代价了。

<a name='5'></a>
## 5 - 代价函数

现在你可以实现前向和反向传播了！你需要计算代价，以便检查你的模型是否真的在学习。

<a name='ex-6'></a>
### 练习 Exercise 6 - compute_cost
用以下公式计算交叉熵代价 $J$：$$-\frac{1}{m} \sum\limits_{i = 1}^{m} (y^{(i)}\log\left(a^{[L] (i)}\right) + (1-y^{(i)})\log\left(1- a^{[L](i)}\right)) \tag{7}$$

In [29]:
# GRADED FUNCTION: compute_cost

def compute_cost(AL, Y):
    """
    Implement the cost function defined by equation (7).

    Arguments:
    AL -- probability vector corresponding to your label predictions, shape (1, number of examples)
    Y -- true "label" vector (for example: containing 0 if non-cat, 1 if cat), shape (1, number of examples)

    Returns:
    cost -- cross-entropy cost
    """
    
    m = Y.shape[1]

    # Compute loss from aL and y.
    # (≈ 1 lines of code)
    # cost = ...
    # YOUR CODE STARTS HERE
    cost = (-1./ m) * np.sum(np.multiply(Y, np.log(AL)) + np.multiply((1-Y), np.log( 1-AL)))
    # YOUR CODE ENDS HERE
    
    cost = np.squeeze(cost)      # To make sure your cost's shape is what we expect (e.g. this turns [[17]] into 17).

    
    return cost

In [30]:
t_Y, t_AL = compute_cost_test_case()
t_cost = compute_cost(t_AL, t_Y)

print("Cost: " + str(t_cost))

compute_cost_test(compute_cost)

Cost: 0.2797765635793422
 All tests passed.


**期望输出**：

<table>
    <tr>
        <td><b>cost</b> </td>
    <td> 0.2797765635793422</td>
    </tr>
</table>

<a name='6'></a>
## 6 - 反向传播模块

正如你为前向传播所做的那样，你会为反向传播实现一些辅助函数。记住，反向传播用来计算损失函数关于参数的梯度。

**提醒**：
<img src="images/backprop_kiank.png" style="width:650px;height:250px;">
<caption><center><font color='purple'><b>图 3</b>：LINEAR->RELU->LINEAR->SIGMOID 的前向与反向传播 <br> <i>紫色块代表前向传播，红色块代表反向传播。</font></center></caption>


<!--
For those of you who are experts in calculus (which you don't need to be to do this assignment!), the chain rule of calculus can be used to derive the derivative of the loss $\mathcal{L}$ with respect to $z^{[1]}$ in a 2-layer network as follows:

$$\frac{d \mathcal{L}(a^{[2]},y)}{{dz^{[1]}}} = \frac{d\mathcal{L}(a^{[2]},y)}{{da^{[2]}}}\frac{{da^{[2]}}}{{dz^{[2]}}}\frac{{dz^{[2]}}}{{da^{[1]}}}\frac{{da^{[1]}}}{{dz^{[1]}}} \tag{8} $$

In order to calculate the gradient $dW^{[1]} = \frac{\partial L}{\partial W^{[1]}}$, use the previous chain rule and you do $dW^{[1]} = dz^{[1]} \times \frac{\partial z^{[1]} }{\partial W^{[1]}}$. During backpropagation, at each step you multiply your current gradient by the gradient corresponding to the specific layer to get the gradient you wanted.

Equivalently, in order to calculate the gradient $db^{[1]} = \frac{\partial L}{\partial b^{[1]}}$, you use the previous chain rule and you do $db^{[1]} = dz^{[1]} \times \frac{\partial z^{[1]} }{\partial b^{[1]}}$.

This is why we talk about **backpropagation**.
!-->

现在，与前向传播类似，你要分三步构建反向传播：
1. LINEAR backward
2. LINEAR -> ACTIVATION backward，其中 ACTIVATION 计算 ReLU 或 sigmoid 激活的导数
3. [LINEAR -> RELU] $\times$ (L-1) -> LINEAR -> SIGMOID backward（整个模型）

对下一个练习，你需要记住：

- `b` 是一个只有 1 列、n 行的矩阵（np.ndarray），即：b = [[1.0], [2.0]]（记住 `b` 是一个常数）
- np.sum 对一个 ndarray 的元素求和
- axis=1 或 axis=0 分别指定求和是按行还是按列进行
- keepdims 指定是否必须保留矩阵的原始维度。
- 看下面的例子来弄清楚：

In [32]:
A = np.array([[1, 2], [3, 4]])

print('axis=1 and keepdims=True')
print(np.sum(A, axis=1, keepdims=True))
print('axis=1 and keepdims=False')
print(np.sum(A, axis=1, keepdims=False))
print('axis=0 and keepdims=True')
print(np.sum(A, axis=0, keepdims=True))
print('axis=0 and keepdims=False')
print(np.sum(A, axis=0, keepdims=False))

axis=1 and keepdims=True
[[3]
 [7]]
axis=1 and keepdims=False
[3 7]
axis=0 and keepdims=True
[[4 6]]
axis=0 and keepdims=False
[4 6]


<a name='6-1'></a>
### 6.1 - Linear Backward

对第 $l$ 层，线性部分是：$Z^{[l]} = W^{[l]} A^{[l-1]} + b^{[l]}$（后接一个激活）。

假设你已经计算了导数 $dZ^{[l]} = \frac{\partial \mathcal{L} }{\partial Z^{[l]}}$。你想得到 $(dW^{[l]}, db^{[l]}, dA^{[l-1]})$。

<img src="images/linearback_kiank.png" style="width:250px;height:300px;">
<caption><center><font color='purple'><b>图 4</b></font></center></caption>

这三个输出 $(dW^{[l]}, db^{[l]}, dA^{[l-1]})$ 用输入 $dZ^{[l]}$ 来计算。

以下是你需要的公式：
$$ dW^{[l]} = \frac{\partial \mathcal{J} }{\partial W^{[l]}} = \frac{1}{m} dZ^{[l]} A^{[l-1] T} \tag{8}$$
$$ db^{[l]} = \frac{\partial \mathcal{J} }{\partial b^{[l]}} = \frac{1}{m} \sum_{i = 1}^{m} dZ^{[l](i)}\tag{9}$$
$$ dA^{[l-1]} = \frac{\partial \mathcal{L} }{\partial A^{[l-1]}} = W^{[l] T} dZ^{[l]} \tag{10}$$


$A^{[l-1] T}$ 是 $A^{[l-1]}$ 的转置。

<a name='ex-7'></a>
### 练习 Exercise 7 - linear_backward

用上面 3 个公式来实现 `linear_backward()`。

**提示**：

- 在 numpy 里，你可以用 `A.T` 或 `A.transpose()` 得到一个 ndarray `A` 的转置

In [33]:
# GRADED FUNCTION: linear_backward

def linear_backward(dZ, cache):
    """
    Implement the linear portion of backward propagation for a single layer (layer l)

    Arguments:
    dZ -- Gradient of the cost with respect to the linear output (of current layer l)
    cache -- tuple of values (A_prev, W, b) coming from the forward propagation in the current layer

    Returns:
    dA_prev -- Gradient of the cost with respect to the activation (of the previous layer l-1), same shape as A_prev
    dW -- Gradient of the cost with respect to W (current layer l), same shape as W
    db -- Gradient of the cost with respect to b (current layer l), same shape as b
    """
    A_prev, W, b = cache
    m = A_prev.shape[1]

    ### START CODE HERE ### (≈ 3 lines of code)
    # dW = ...
    # db = ... sum by the rows of dZ with keepdims=True
    # dA_prev = ...
    # YOUR CODE STARTS HERE
    dW = (1./m) * np.dot(dZ,cache[0].T)
    db = (1./m) * np.sum(dZ,axis=1,keepdims=True)
    dA_prev = np.dot(cache[1].T,dZ)
    # YOUR CODE ENDS HERE
    
    return dA_prev, dW, db

In [35]:
t_dZ, t_linear_cache = linear_backward_test_case()
t_dA_prev, t_dW, t_db = linear_backward(t_dZ, t_linear_cache)

print("dA_prev: " + str(t_dA_prev))
print("dW: " + str(t_dW))
print("db: " + str(t_db))

linear_backward_test(linear_backward)

dA_prev: [[-1.15171336  0.06718465 -0.3204696   2.09812712]
 [ 0.60345879 -3.72508701  5.81700741 -3.84326836]
 [-0.4319552  -1.30987417  1.72354705  0.05070578]
 [-0.38981415  0.60811244 -1.25938424  1.47191593]
 [-2.52214926  2.67882552 -0.67947465  1.48119548]]
dW: [[ 0.07313866 -0.0976715  -0.87585828  0.73763362  0.00785716]
 [ 0.85508818  0.37530413 -0.59912655  0.71278189 -0.58931808]
 [ 0.97913304 -0.24376494 -0.08839671  0.55151192 -0.10290907]]
db: [[-0.14713786]
 [-0.11313155]
 [-0.13209101]]
 All tests passed.


**期望输出**：
```
dA_prev: [[-1.15171336  0.06718465 -0.3204696   2.09812712]
 [ 0.60345879 -3.72508701  5.81700741 -3.84326836]
 [-0.4319552  -1.30987417  1.72354705  0.05070578]
 [-0.38981415  0.60811244 -1.25938424  1.47191593]
 [-2.52214926  2.67882552 -0.67947465  1.48119548]]
dW: [[ 0.07313866 -0.0976715  -0.87585828  0.73763362  0.00785716]
 [ 0.85508818  0.37530413 -0.59912655  0.71278189 -0.58931808]
 [ 0.97913304 -0.24376494 -0.08839671  0.55151192 -0.10290907]]
db: [[-0.14713786]
 [-0.11313155]
 [-0.13209101]]
 ```

<a name='6-2'></a>
### 6.2 - Linear-Activation Backward

接下来，你会创建一个函数，把两个辅助函数合并起来：**`linear_backward`** 和激活的反向步骤 **`linear_activation_backward`**。

为帮你实现 `linear_activation_backward`，已提供两个反向函数：
- **`sigmoid_backward`**：实现 SIGMOID 单元的反向传播。你可以这样调用它：

```python
dZ = sigmoid_backward(dA, activation_cache)
```

- **`relu_backward`**：实现 RELU 单元的反向传播。你可以这样调用它：

```python
dZ = relu_backward(dA, activation_cache)
```

若 $g(.)$ 是激活函数，
`sigmoid_backward` 和 `relu_backward` 计算 $$dZ^{[l]} = dA^{[l]} * g'(Z^{[l]}). \tag{11}$$

<a name='ex-8'></a>
### 练习 Exercise 8 - linear_activation_backward

实现 *LINEAR->ACTIVATION* 层的反向传播。

In [38]:
# GRADED FUNCTION: linear_activation_backward

def linear_activation_backward(dA, cache, activation):
    """
    Implement the backward propagation for the LINEAR->ACTIVATION layer.
    
    Arguments:
    dA -- post-activation gradient for current layer l 
    cache -- tuple of values (linear_cache, activation_cache) we store for computing backward propagation efficiently
    activation -- the activation to be used in this layer, stored as a text string: "sigmoid" or "relu"
    
    Returns:
    dA_prev -- Gradient of the cost with respect to the activation (of the previous layer l-1), same shape as A_prev
    dW -- Gradient of the cost with respect to W (current layer l), same shape as W
    db -- Gradient of the cost with respect to b (current layer l), same shape as b
    """
    linear_cache, activation_cache = cache
    
    if activation == "relu":
        #(≈ 2 lines of code)
        # dZ =  ...
        # dA_prev, dW, db =  ...
        # YOUR CODE STARTS HERE
        dZ = relu_backward(dA,activation_cache)
        dA_prev, dW, db = linear_backward(dZ,linear_cache)
        # YOUR CODE ENDS HERE
        
    elif activation == "sigmoid":
        #(≈ 2 lines of code)
        # dZ =  ...
        # dA_prev, dW, db =  ...
        # YOUR CODE STARTS HERE
        dZ = sigmoid_backward(dA,activation_cache)
        dA_prev, dW, db = linear_backward(dZ,linear_cache)
        # YOUR CODE ENDS HERE
    
    return dA_prev, dW, db

In [39]:
t_dAL, t_linear_activation_cache = linear_activation_backward_test_case()

t_dA_prev, t_dW, t_db = linear_activation_backward(t_dAL, t_linear_activation_cache, activation = "sigmoid")
print("With sigmoid: dA_prev = " + str(t_dA_prev))
print("With sigmoid: dW = " + str(t_dW))
print("With sigmoid: db = " + str(t_db))

t_dA_prev, t_dW, t_db = linear_activation_backward(t_dAL, t_linear_activation_cache, activation = "relu")
print("With relu: dA_prev = " + str(t_dA_prev))
print("With relu: dW = " + str(t_dW))
print("With relu: db = " + str(t_db))

linear_activation_backward_test(linear_activation_backward)

With sigmoid: dA_prev = [[ 0.11017994  0.01105339]
 [ 0.09466817  0.00949723]
 [-0.05743092 -0.00576154]]
With sigmoid: dW = [[ 0.10266786  0.09778551 -0.01968084]]
With sigmoid: db = [[-0.05729622]]
With relu: dA_prev = [[ 0.44090989  0.        ]
 [ 0.37883606  0.        ]
 [-0.2298228   0.        ]]
With relu: dW = [[ 0.44513824  0.37371418 -0.10478989]]
With relu: db = [[-0.20837892]]
 All tests passed.


**期望输出：**

```
With sigmoid: dA_prev = [[ 0.11017994  0.01105339]
 [ 0.09466817  0.00949723]
 [-0.05743092 -0.00576154]]
With sigmoid: dW = [[ 0.10266786  0.09778551 -0.01968084]]
With sigmoid: db = [[-0.05729622]]
With relu: dA_prev = [[ 0.44090989  0.        ]
 [ 0.37883606  0.        ]
 [-0.2298228   0.        ]]
With relu: dW = [[ 0.44513824  0.37371418 -0.10478989]]
With relu: db = [[-0.20837892]]
```

<a name='6-3'></a>
### 6.3 - L-Model Backward

现在你要实现整个网络的反向函数了！

回忆一下，当你实现 `L_model_forward` 函数时，每次迭代你都存了一个包含 (X,W,b, 和 z) 的 cache。在反向传播模块里，你会用这些变量来计算梯度。因此，在 `L_model_backward` 函数里，你会从第 $L$ 层开始，反向遍历所有隐藏层。每一步，你会用第 $l$ 层的缓存值来反向传播穿过第 $l$ 层。下面的图 5 展示了反向传递。

<img src="images/mn_backward.png" style="width:450px;height:300px;">
<caption><center><font color='purple'><b>图 5</b>：反向传递</font></center></caption>

**初始化反向传播**：

要反向传播穿过这个网络，你知道输出是：
$A^{[L]} = \sigma(Z^{[L]})$。所以你的代码需要计算 `dAL` $= \frac{\partial \mathcal{L}}{\partial A^{[L]}}$。
为此，用这个公式（用微积分推导，同样你不需要对它有深入了解！）：
```python
dAL = - (np.divide(Y, AL) - np.divide(1 - Y, 1 - AL)) # derivative of cost with respect to AL
```

然后你可以用这个激活后的梯度 `dAL` 继续往回走。如图 5 所示，你现在可以把 `dAL` 喂给你实现的 LINEAR->SIGMOID 反向函数（它会用到 L_model_forward 函数存的缓存值）。

之后，你要用一个 `for` 循环，用 LINEAR->RELU 反向函数遍历所有其他层。你应该把每个 dA、dW 和 db 存进 grads 字典。为此，用这个公式：

$$grads["dW" + str(l)] = dW^{[l]}\tag{15} $$

例如，对 $l=3$，这会把 $dW^{[l]}$ 存进 `grads["dW3"]`。

<a name='ex-9'></a>
### 练习 Exercise 9 - L_model_backward

为 *[LINEAR->RELU] $\times$ (L-1) -> LINEAR -> SIGMOID* 模型实现反向传播。

In [46]:
# GRADED FUNCTION: L_model_backward

def L_model_backward(AL, Y, caches):
    """
    Implement the backward propagation for the [LINEAR->RELU] * (L-1) -> LINEAR -> SIGMOID group
    
    Arguments:
    AL -- probability vector, output of the forward propagation (L_model_forward())
    Y -- true "label" vector (containing 0 if non-cat, 1 if cat)
    caches -- list of caches containing:
                every cache of linear_activation_forward() with "relu" (it's caches[l], for l in range(L-1) i.e l = 0...L-2)
                the cache of linear_activation_forward() with "sigmoid" (it's caches[L-1])
    
    Returns:
    grads -- A dictionary with the gradients
             grads["dA" + str(l)] = ... 
             grads["dW" + str(l)] = ...
             grads["db" + str(l)] = ... 
    """
    grads = {}
    L = len(caches) # the number of layers
    m = AL.shape[1]
    Y = Y.reshape(AL.shape) # after this line, Y is the same shape as AL
    
    # Initializing the backpropagation
    #(1 line of code)
    # dAL = ...
    # YOUR CODE STARTS HERE
    dAL = - (np.divide(Y, AL) - np.divide(1 - Y, 1 - AL))
    # YOUR CODE ENDS HERE
    
    # Lth layer (SIGMOID -> LINEAR) gradients. Inputs: "dAL, current_cache". Outputs: "grads["dAL-1"], grads["dWL"], grads["dbL"]
    #(approx. 5 lines)
    # current_cache = ...
    # dA_prev_temp, dW_temp, db_temp = ...
    # grads["dA" + str(L-1)] = ...
    # grads["dW" + str(L)] = ...
    # grads["db" + str(L)] = ...
    # YOUR CODE STARTS HERE
    current_cache = caches[-1]
    grads["dA" + str(L-1)], grads["dW" + str(L)], grads["db" + str(L)] = linear_activation_backward(dAL, current_cache, activation="sigmoid")
    # YOUR CODE ENDS HERE
    
    # Loop from l=L-2 to l=0
    for l in reversed(range(L-1)):
        # lth layer: (RELU -> LINEAR) gradients.
        # Inputs: "grads["dA" + str(l + 1)], current_cache". Outputs: "grads["dA" + str(l)] , grads["dW" + str(l + 1)] , grads["db" + str(l + 1)] 
        #(approx. 5 lines)
        # current_cache = ...
        # dA_prev_temp, dW_temp, db_temp = ...
        # grads["dA" + str(l)] = ...
        # grads["dW" + str(l + 1)] = ...
        # grads["db" + str(l + 1)] = ...
        # YOUR CODE STARTS HERE
        current_cache = caches[l]
        dA_prev_temp, dW_temp, db_temp = linear_activation_backward(grads["dA" + str(l + 1)], current_cache, activation="relu")
        grads["dA" + str(l)] = dA_prev_temp
        grads["dW" + str(l + 1)] = dW_temp
        grads["db" + str(l + 1)] = db_temp
        # YOUR CODE ENDS HERE

    return grads

In [47]:
t_AL, t_Y_assess, t_caches = L_model_backward_test_case()
grads = L_model_backward(t_AL, t_Y_assess, t_caches)

print("dA0 = " + str(grads['dA0']))
print("dA1 = " + str(grads['dA1']))
print("dW1 = " + str(grads['dW1']))
print("dW2 = " + str(grads['dW2']))
print("db1 = " + str(grads['db1']))
print("db2 = " + str(grads['db2']))

L_model_backward_test(L_model_backward)

dA0 = [[ 0.          0.52257901]
 [ 0.         -0.3269206 ]
 [ 0.         -0.32070404]
 [ 0.         -0.74079187]]
dA1 = [[ 0.12913162 -0.44014127]
 [-0.14175655  0.48317296]
 [ 0.01663708 -0.05670698]]
dW1 = [[0.41010002 0.07807203 0.13798444 0.10502167]
 [0.         0.         0.         0.        ]
 [0.05283652 0.01005865 0.01777766 0.0135308 ]]
dW2 = [[-0.39202432 -0.13325855 -0.04601089]]
db1 = [[-0.22007063]
 [ 0.        ]
 [-0.02835349]]
db2 = [[0.15187861]]
 All tests passed.


**期望输出：**

```
dA0 = [[ 0.          0.52257901]
 [ 0.         -0.3269206 ]
 [ 0.         -0.32070404]
 [ 0.         -0.74079187]]
dA1 = [[ 0.12913162 -0.44014127]
 [-0.14175655  0.48317296]
 [ 0.01663708 -0.05670698]]
dW1 = [[0.41010002 0.07807203 0.13798444 0.10502167]
 [0.         0.         0.         0.        ]
 [0.05283652 0.01005865 0.01777766 0.0135308 ]]
dW2 = [[-0.39202432 -0.13325855 -0.04601089]]
db1 = [[-0.22007063]
 [ 0.        ]
 [-0.02835349]]
db2 = [[0.15187861]]
```

<a name='6-4'></a>
### 6.4 - 更新参数

在这一节，你会用梯度下降更新模型的参数：

$$ W^{[l]} = W^{[l]} - \alpha \text{ } dW^{[l]} \tag{16}$$
$$ b^{[l]} = b^{[l]} - \alpha \text{ } db^{[l]} \tag{17}$$

其中 $\alpha$ 是学习率。

算出更新后的参数后，把它们存进 parameters 字典。

<a name='ex-10'></a>
### 练习 Exercise 10 - update_parameters

实现 `update_parameters()`，用梯度下降来更新你的参数。

**说明**：
对每个 $W^{[l]}$ 和 $b^{[l]}$（$l = 1, 2, ..., L$）用梯度下降更新参数。

In [48]:
# GRADED FUNCTION: update_parameters

def update_parameters(params, grads, learning_rate):
    """
    Update parameters using gradient descent
    
    Arguments:
    params -- python dictionary containing your parameters 
    grads -- python dictionary containing your gradients, output of L_model_backward
    
    Returns:
    parameters -- python dictionary containing your updated parameters 
                  parameters["W" + str(l)] = ... 
                  parameters["b" + str(l)] = ...
    """
    parameters = copy.deepcopy(params)
    L = len(parameters) // 2 # number of layers in the neural network

    # Update rule for each parameter. Use a for loop.
    #(≈ 2 lines of code)
    for l in range(L):
        # parameters["W" + str(l+1)] = ...
        # parameters["b" + str(l+1)] = ...
        # YOUR CODE STARTS HERE
        parameters["W"+str(l+1)] -= learning_rate * grads["dW" + str(l+1)]
        parameters["b"+str(l+1)] -= learning_rate * grads["db" + str(l+1)]
        # YOUR CODE ENDS HERE
    return parameters

In [49]:
t_parameters, grads = update_parameters_test_case()
t_parameters = update_parameters(t_parameters, grads, 0.1)

print ("W1 = "+ str(t_parameters["W1"]))
print ("b1 = "+ str(t_parameters["b1"]))
print ("W2 = "+ str(t_parameters["W2"]))
print ("b2 = "+ str(t_parameters["b2"]))

update_parameters_test(update_parameters)

W1 = [[-0.59562069 -0.09991781 -2.14584584  1.82662008]
 [-1.76569676 -0.80627147  0.51115557 -1.18258802]
 [-1.0535704  -0.86128581  0.68284052  2.20374577]]
b1 = [[-0.04659241]
 [-1.28888275]
 [ 0.53405496]]
W2 = [[-0.55569196  0.0354055   1.32964895]]
b2 = [[-0.84610769]]
 All tests passed.


**期望输出：**

```
W1 = [[-0.59562069 -0.09991781 -2.14584584  1.82662008]
 [-1.76569676 -0.80627147  0.51115557 -1.18258802]
 [-1.0535704  -0.86128581  0.68284052  2.20374577]]
b1 = [[-0.04659241]
 [-1.28888275]
 [ 0.53405496]]
W2 = [[-0.55569196  0.0354055   1.32964895]]
b2 = [[-0.84610769]]
```

### 恭喜！

你刚刚实现了构建一个深度神经网络所需的所有函数，包括：

- 使用非线性单元改进你的模型
- 构建一个更深的神经网络（多于 1 个隐藏层）
- 实现一个易于使用的神经网络类

这确实是个很长的作业，但作业的下一部分会更容易。;)

在下一个作业里，你会把这些全部拼到一起，构建两个模型：

- 一个两层神经网络
- 一个 L 层神经网络

你实际上会用这些模型来分类猫 vs 非猫的图像！（喵！）干得漂亮，下次见。